# UFD CelebDFv1 Corruption Test

Train on pre-extracted FF++ features, then test CelebDFv1 corruption features for one selected level.


In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from tqdm import tqdm


In [2]:
FEATURE_DIR = Path("/kaggle/input/datasets/vhonghoavin/deepfakebench-features")
LEVEL = 3

TRAIN_FEATURE_PATH = FEATURE_DIR / "ffpp_train_features.pt"
CORRUPTIONS = [
    "color_contrast",
    "color_saturation",
    "gaussian_blur",
    "resize",
]

MODEL_OUTPUT_PATH = f"/kaggle/working/ufd_linear_probe_ffpp_level{LEVEL}.pt"
RESULTS_OUTPUT_PATH = f"/kaggle/working/celebdfv1_level{LEVEL}_corruption_results.csv"

EPOCHS = 200
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 4096
TIP_TEST_BATCH_SIZE = 512
TIP_CACHE_BATCH_SIZE = 8192
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE


'cuda'

## Load Features


In [3]:
def corruption_feature_path(corruption, level=LEVEL):
    return FEATURE_DIR / f"celebdfv1_level{level}_{corruption}_features.pt"


def load_feature_file(path):
    payload = torch.load(path, map_location="cpu")
    features = payload["features"].float()
    labels = payload["labels"].long()

    print(path)
    print("features:", tuple(features.shape))
    print("labels:", tuple(labels.shape))
    print("label counts [REAL, FAKE]:", torch.bincount(labels, minlength=2).tolist())
    print("dataset:", payload.get("dataset_name", payload.get("target_dataset", "unknown")))
    print("transform:", payload.get("transform_name", "none"))
    print("clip:", payload.get("clip_model", "unknown"))
    print()

    return features, labels


train_feats, train_labels = load_feature_file(TRAIN_FEATURE_PATH)

test_features = {}
for corruption in CORRUPTIONS:
    feats, labels = load_feature_file(corruption_feature_path(corruption))
    test_features[corruption] = {"features": feats, "labels": labels}


/kaggle/input/datasets/vhonghoavin/deepfakebench-features/ffpp_train_features.pt
features: (513568, 768)
labels: (513568,)
label counts [REAL, FAKE]: [42690, 470878]
dataset: FaceForensics++
transform: none
clip: ViT-L-14/openai

/kaggle/input/datasets/vhonghoavin/deepfakebench-features/celebdfv1_level3_color_contrast_features.pt
features: (38312, 768)
labels: (38312,)
label counts [REAL, FAKE]: [5004, 33308]
dataset: Celeb-DF-v1
transform: color_contrast
clip: ViT-L-14/openai

/kaggle/input/datasets/vhonghoavin/deepfakebench-features/celebdfv1_level3_color_saturation_features.pt
features: (38312, 768)
labels: (38312,)
label counts [REAL, FAKE]: [5004, 33308]
dataset: Celeb-DF-v1
transform: color_saturation
clip: ViT-L-14/openai

/kaggle/input/datasets/vhonghoavin/deepfakebench-features/celebdfv1_level3_gaussian_blur_features.pt
features: (38312, 768)
labels: (38312,)
label counts [REAL, FAKE]: [5004, 33308]
dataset: Celeb-DF-v1
transform: gaussian_blur
clip: ViT-L-14/openai

/kaggle/i

## Train Linear Probe


In [4]:
class LinearProbe(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(1)


clf = LinearProbe(train_feats.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(clf.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

X_train = train_feats.to(DEVICE)
y_train = train_labels.float().to(DEVICE)


In [5]:
for epoch in range(EPOCHS):
    clf.train()

    permutation = torch.randperm(len(X_train), device=DEVICE)
    total_loss = 0.0
    total_correct = 0
    total = 0

    for start in range(0, len(X_train), TRAIN_BATCH_SIZE):
        idx = permutation[start:start + TRAIN_BATCH_SIZE]
        xb = X_train[idx]
        yb = y_train[idx]

        logits = clf(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = len(xb)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        total_loss += loss.item() * batch_size
        total_correct += (preds == yb.long()).sum().item()
        total += batch_size

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"loss={total_loss / total:.4f} | "
        f"train_acc={total_correct / total:.4f}"
    )

torch.save(clf.state_dict(), MODEL_OUTPUT_PATH)
print("saved model to:", MODEL_OUTPUT_PATH)


Epoch 1/200 | loss=0.2513 | train_acc=0.9152
Epoch 2/200 | loss=0.1983 | train_acc=0.9170
Epoch 3/200 | loss=0.1875 | train_acc=0.9176
Epoch 4/200 | loss=0.1814 | train_acc=0.9188
Epoch 5/200 | loss=0.1773 | train_acc=0.9201
Epoch 6/200 | loss=0.1743 | train_acc=0.9212
Epoch 7/200 | loss=0.1720 | train_acc=0.9222
Epoch 8/200 | loss=0.1701 | train_acc=0.9231
Epoch 9/200 | loss=0.1684 | train_acc=0.9238
Epoch 10/200 | loss=0.1670 | train_acc=0.9244
Epoch 11/200 | loss=0.1658 | train_acc=0.9250
Epoch 12/200 | loss=0.1648 | train_acc=0.9255
Epoch 13/200 | loss=0.1638 | train_acc=0.9260
Epoch 14/200 | loss=0.1629 | train_acc=0.9264
Epoch 15/200 | loss=0.1621 | train_acc=0.9269
Epoch 16/200 | loss=0.1614 | train_acc=0.9273
Epoch 17/200 | loss=0.1608 | train_acc=0.9276
Epoch 18/200 | loss=0.1602 | train_acc=0.9279
Epoch 19/200 | loss=0.1596 | train_acc=0.9281
Epoch 20/200 | loss=0.1591 | train_acc=0.9284
Epoch 21/200 | loss=0.1586 | train_acc=0.9287
Epoch 22/200 | loss=0.1581 | train_acc=0.92

## Metrics


In [6]:
def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[eer_idx] + fnr[eer_idx]) / 2, thresholds[eer_idx]


def evaluate_scores(y_true, y_score, y_pred, name="test", show_report=True):
    eer, eer_threshold = calculate_eer(y_true, y_score)
    metrics = {
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro"),
        "auc": roc_auc_score(y_true, y_score),
        "ap": average_precision_score(y_true, y_score),
        "eer": eer,
        "eer_threshold": eer_threshold,
    }

    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"{key}: {value}")

    if show_report:
        print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"]))

    return metrics


@torch.no_grad()
def evaluate_probe(model, feats, labels, name="test", batch_size=EVAL_BATCH_SIZE, show_report=True):
    model.eval()
    scores = []

    for start in range(0, len(feats), batch_size):
        xb = feats[start:start + batch_size].to(DEVICE)
        scores.append(torch.sigmoid(model(xb)).cpu())

    y_score = torch.cat(scores).numpy()
    y_true = labels.numpy()
    y_pred = (y_score >= 0.5).astype(int)
    return evaluate_scores(y_true, y_score, y_pred, name=name, show_report=show_report)


## Tip-Adapter


In [7]:
def build_tip_cache(train_feats, train_labels, num_classes=2):
    cache_keys = F.normalize(train_feats.float(), dim=-1)
    cache_values = F.one_hot(train_labels.long(), num_classes=num_classes).float()
    return cache_keys, cache_values


@torch.no_grad()
def tip_adapter_predict(
    model,
    test_feats,
    cache_keys,
    cache_values,
    alpha=0.5,
    beta=5.5,
    test_batch_size=TIP_TEST_BATCH_SIZE,
    cache_batch_size=TIP_CACHE_BATCH_SIZE,
):
    model.eval()
    cache_keys = cache_keys.to(DEVICE)
    cache_values = cache_values.to(DEVICE)
    probs = []

    for start in tqdm(range(0, len(test_feats), test_batch_size)):
        feats = test_feats[start:start + test_batch_size].float().to(DEVICE)
        feats = F.normalize(feats, dim=-1)

        base_logits = model(feats)
        p_fake = torch.sigmoid(base_logits)
        p_base = torch.stack([1 - p_fake, p_fake], dim=1)

        cache_logits = torch.zeros((len(feats), cache_values.shape[1]), device=DEVICE)
        cache_norm = torch.zeros((len(feats), 1), device=DEVICE)

        for cache_start in range(0, len(cache_keys), cache_batch_size):
            keys = cache_keys[cache_start:cache_start + cache_batch_size]
            values = cache_values[cache_start:cache_start + cache_batch_size]
            similarity = feats @ keys.T
            affinity = torch.exp(-beta * (1 - similarity))
            cache_logits += affinity @ values
            cache_norm += affinity.sum(dim=1, keepdim=True)

        p_cache = cache_logits / (cache_norm + 1e-8)
        probs.append((alpha * p_cache + (1 - alpha) * p_base).cpu())

    return torch.cat(probs, dim=0)


@torch.no_grad()
def evaluate_tip_adapter(
    model,
    test_feats,
    test_labels,
    cache_keys,
    cache_values,
    name="test",
    alpha=0.5,
    beta=5.5,
    show_report=True,
):
    probs = tip_adapter_predict(
        model=model,
        test_feats=test_feats,
        cache_keys=cache_keys,
        cache_values=cache_values,
        alpha=alpha,
        beta=beta,
    )

    print(f"alpha: {alpha} | beta: {beta}")
    metrics = evaluate_scores(
        y_true=test_labels.numpy(),
        y_score=probs[:, 1].numpy(),
        y_pred=probs.argmax(dim=1).numpy(),
        name=name,
        show_report=show_report,
    )
    return metrics


cache_keys, cache_values = build_tip_cache(train_feats, train_labels)


## Test All Corruptions


In [8]:
rows = []

train_metrics = evaluate_probe(clf, train_feats, train_labels, name="Train FF++", show_report=False)
rows.append({"level": LEVEL, "corruption": "train_ffpp", "method": "linear_probe", **train_metrics})

for corruption, payload in test_features.items():
    feats = payload["features"]
    labels = payload["labels"]

    probe_metrics = evaluate_probe(
        clf,
        feats,
        labels,
        name=f"CelebDFv1 level {LEVEL} {corruption} | Linear Probe",
        show_report=False,
    )
    rows.append({"level": LEVEL, "corruption": corruption, "method": "linear_probe", **probe_metrics})

    tip_metrics = evaluate_tip_adapter(
        clf,
        feats,
        labels,
        cache_keys,
        cache_values,
        name=f"CelebDFv1 level {LEVEL} {corruption} | Tip-Adapter",
        alpha=0.5,
        beta=5.5,
        show_report=False,
    )
    rows.append({"level": LEVEL, "corruption": corruption, "method": "tip_adapter", **tip_metrics})

results_df = pd.DataFrame(rows)
results_df



Train FF++
acc: 0.9347077699545143
f1: 0.7338796906146711
auc: 0.9485726808818782
ap: 0.9951740228849315
eer: 0.12486229069020904
eer_threshold: 0.865460991859436

CelebDFv1 level 3 color_contrast | Linear Probe
acc: 0.8443307579870537
f1: 0.5299616107394479
auc: 0.6972870064702411
ap: 0.9364932739204557
eer: 0.3525079420071484
eer_threshold: 0.9511111378669739


100%|██████████| 75/75 [00:11<00:00,  6.65it/s]


alpha: 0.5 | beta: 5.5

CelebDFv1 level 3 color_contrast | Tip-Adapter
acc: 0.8664387137189392
f1: 0.47728145406475475
auc: 0.6787481387533183
ap: 0.9231597600529731
eer: 0.360422722228126
eer_threshold: 0.9308227300643921

CelebDFv1 level 3 color_saturation | Linear Probe
acc: 0.855449989559407
f1: 0.5219301736918815
auc: 0.6438327061420397
ap: 0.9195038456390102
eer: 0.3966720582942797
eer_threshold: 0.9680941104888916


100%|██████████| 75/75 [00:11<00:00,  6.64it/s]


alpha: 0.5 | beta: 5.5

CelebDFv1 level 3 color_saturation | Tip-Adapter
acc: 0.8676132804343286
f1: 0.4717106023028489
auc: 0.6136530129805127
ap: 0.9031197838681592
eer: 0.41151180172710633
eer_threshold: 0.9357161521911621

CelebDFv1 level 3 gaussian_blur | Linear Probe
acc: 0.8487158070578409
f1: 0.5184656849444823
auc: 0.5685088173006688
ap: 0.884902171342298
eer: 0.44944101161967026
eer_threshold: 0.9577988386154175


100%|██████████| 75/75 [00:11<00:00,  6.49it/s]


alpha: 0.5 | beta: 5.5

CelebDFv1 level 3 gaussian_blur | Tip-Adapter
acc: 0.867273961160994
f1: 0.4675708725717931
auc: 0.5515435405968487
ap: 0.875240008047171
eer: 0.4594057910870775
eer_threshold: 0.9371029138565063

CelebDFv1 level 3 resize | Linear Probe
acc: 0.8581384422635205
f1: 0.5115085982810863
auc: 0.6043617009838749
ap: 0.9082589809237851
eer: 0.4308864065226743
eer_threshold: 0.9723325967788696


100%|██████████| 75/75 [00:11<00:00,  6.25it/s]


alpha: 0.5 | beta: 5.5

CelebDFv1 level 3 resize | Tip-Adapter
acc: 0.8677437878471497
f1: 0.4659654120145581
auc: 0.5685675099886465
ap: 0.8856745896036136
eer: 0.4469261266860176
eer_threshold: 0.9420993328094482


,level,corruption,method,acc,f1,auc,ap,eer,eer_threshold
0,3,train_ffpp,linear_probe,0.934708,0.733880,0.948573,0.995174,0.124862,0.865461
1,3,color_contrast,linear_probe,0.844331,0.529962,0.697287,0.936493,0.352508,0.951111
2,3,color_contrast,tip_adapter,0.866439,0.477281,0.678748,0.923160,0.360423,0.930823
3,3,color_saturation,linear_probe,0.855450,0.521930,0.643833,0.919504,0.396672,0.968094
4,3,color_saturation,tip_adapter,0.867613,0.471711,0.613653,0.903120,0.411512,0.935716
5,3,gaussian_blur,linear_probe,0.848716,0.518466,0.568509,0.884902,0.449441,0.957799
6,3,gaussian_blur,tip_adapter,0.867274,0.467571,0.551544,0.875240,0.459406,0.937103
7,3,resize,linear_probe,0.858138,0.511509,0.604362,0.908259,0.430886,0.972333
8,3,resize,tip_adapter,0.867744,0.465965,0.568568,0.885675,0.446926,0.942099


In [9]:
results_df.to_csv(RESULTS_OUTPUT_PATH, index=False)
print("saved results to:", RESULTS_OUTPUT_PATH)
results_df


saved results to: /kaggle/working/celebdfv1_level3_corruption_results.csv


,level,corruption,method,acc,f1,auc,ap,eer,eer_threshold
0,3,train_ffpp,linear_probe,0.934708,0.733880,0.948573,0.995174,0.124862,0.865461
1,3,color_contrast,linear_probe,0.844331,0.529962,0.697287,0.936493,0.352508,0.951111
2,3,color_contrast,tip_adapter,0.866439,0.477281,0.678748,0.923160,0.360423,0.930823
3,3,color_saturation,linear_probe,0.855450,0.521930,0.643833,0.919504,0.396672,0.968094
4,3,color_saturation,tip_adapter,0.867613,0.471711,0.613653,0.903120,0.411512,0.935716
5,3,gaussian_blur,linear_probe,0.848716,0.518466,0.568509,0.884902,0.449441,0.957799
6,3,gaussian_blur,tip_adapter,0.867274,0.467571,0.551544,0.875240,0.459406,0.937103
7,3,resize,linear_probe,0.858138,0.511509,0.604362,0.908259,0.430886,0.972333
8,3,resize,tip_adapter,0.867744,0.465965,0.568568,0.885675,0.446926,0.942099
